# Análise de Dados e Visualização
## Fundamentos de Análise de Dados

**Autor:** Victor Henriques, Ph.D.

Este notebook apresenta fundamentos de inspeção, transformação, agregação, reorganização e combinação de dados com `pandas`.

## Objetivos de aprendizagem

Ao final deste módulo, os estudantes serão capazes de:

- inspecionar a estrutura, os tipos e os valores ausentes de um DataFrame;
- selecionar, filtrar, ordenar e reorganizar linhas e colunas;
- criar variáveis com operações vetorizadas e encadeamento de métodos;
- resumir dados com agrupamentos e agregações nomeadas;
- gerenciar índices e MultiIndexes;
- reorganizar dados entre os formatos largo e longo;
- combinar conjuntos de dados por concatenação e junções relacionais;
- preparar dados reproduzíveis e prontos para análise e visualização.

Uma visualização confiável começa com dados corretamente estruturados e transformados.

## Configuração do ambiente

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)

# 1. DataFrames

Um `DataFrame` é o principal objeto tabular do `pandas`. Ele organiza informações em linhas e colunas. Normalmente, cada linha representa uma observação e cada coluna representa uma variável. As colunas podem armazenar diferentes tipos de dados. O índice identifica e organiza as linhas, enquanto os nomes das colunas identificam as variáveis.

In [ ]:
df_basico = pd.DataFrame({
    "col0": [0, 0, 0, 0],
    "col1": [0, 0, 0, 0],
    "col2": [0, 0, 0, 0],
    "col3": ["a", "b", "b", "a"],
    "col4": ["alpha", "gamma", "gamma", "gamma"],
}, index=[f"row{i}" for i in range(4)])

df_basico.head()

O argumento `data` recebe um dicionário: cada chave representa uma coluna e cada lista contém seus valores. O índice aparece à esquerda, mas não é uma coluna comum. Como a tabela possui quatro linhas, `head()` apresenta todas elas.

# 2. Base de dados de voos

A base `voos` contém informações sobre voos que partiram de JFK, LaGuardia e Newark em 2013. Cada linha representa um voo individual. As colunas descrevem datas, horários, atrasos, companhia aérea, origem, destino, distância e tempo de voo.

In [ ]:
url_voos = "https://raw.githubusercontent.com/byuidatascience/data4python4ds/master/data-raw/flights/flights.csv"
voos = pd.read_csv(url_voos)
voos.head()

## Inspeção da estrutura

`info()` apresenta número de linhas, índice, nomes das colunas, quantidade de valores não nulos, tipos de dados e uso aproximado de memória. Se a quantidade de valores não nulos for menor que o total de linhas, há valores ausentes.

In [ ]:
voos.info()

## Conversão de data e hora

A coluna `time_hour` parece conter datas, mas inicialmente é texto. `pd.to_datetime()` converte os valores para um tipo temporal. O argumento `utc=True` preserva a referência UTC indicada por `Z`.

In [ ]:
voos["time_hour"].head()

In [ ]:
voos["time_hour"] = pd.to_datetime(
    voos["time_hour"],
    format="%Y-%m-%dT%H:%M:%SZ",
    utc=True,
)
voos["time_hour"].head()

# 3. Manipulação de linhas

Usaremos uma tabela pequena para distinguir rótulos, posições e condições.

In [ ]:
df = pd.DataFrame(
    np.arange(36).reshape(6, 6),
    index=list("abcdef"),
    columns=[f"col{i}" for i in range(6)],
    dtype=float,
)
df["col6"] = ["apple", "orange", "pineapple", "mango", "kiwi", "lemon"]
df

## Seleção com `.loc` e `.iloc`

`.loc` seleciona por rótulos do índice. `.iloc` seleciona por posições inteiras iniciadas em zero. Uma lista preserva o resultado como DataFrame.

In [ ]:
df.loc[["a", "b"]]

In [ ]:
df.iloc[[0, 2]]

## Filtragem com `query()`

`query()` filtra linhas a partir de uma condição textual. Podemos usar `and`, `or`, `>`, `>=`, `<`, `<=`, `==` e `!=`. `==` compara valores; `=` atribui valores.

In [ ]:
df.query("col6 == 'kiwi' or col6 == 'pineapple'")

In [ ]:
df.query("col0 > 6")

## Ordenação

`sort_values()` reorganiza linhas por uma ou mais colunas. Colunas posteriores resolvem empates nas anteriores. `ascending=False` produz ordem decrescente. Valores ausentes ficam no final por padrão.

In [ ]:
voos.sort_values(["year", "month", "day", "dep_time"]).head(3)

In [ ]:
voos.sort_values("dep_delay", ascending=False).head(3)

## Encadeamento de operações

O pipeline abaixo filtra voos que partiram aproximadamente no horário, ordena pelo atraso de chegada e seleciona os três maiores casos.

In [ ]:
(
    voos
    .query("dep_delay <= 10 and dep_delay >= -10")
    .sort_values("arr_delay", ascending=False)
    .iloc[:3]
)

# 4. Manipulação de colunas

## Criando colunas

Um escalar é repetido em todas as linhas. Uma nova atribuição à mesma coluna substitui os valores anteriores. Uma sequência deve ter comprimento compatível com o número de linhas.

In [ ]:
df["new_column0"] = 5
df.head(3)

In [ ]:
df["new_column0"] = [0, 1, 2, 3, 4, 5]
df.head(3)

## Variáveis derivadas

Operações vetorizadas atuam sobre colunas inteiras e dispensam loops explícitos.

In [ ]:
df[["new_column1", "new_column2"]] = [5, 6]
df["new_column3"] = df["col0"] - df["new_column0"]
df.insert(0, "new_column3", df.pop("new_column3"))
df.head()

## Criando variáveis com `assign()`

`assign()` retorna um novo DataFrame. Cada `lambda` recebe o DataFrame completo. `gain` mede a recuperação líquida de atraso; `speed_mph` mede a velocidade média em milhas por hora.

In [ ]:
voos_enriquecidos = voos.assign(
    gain=lambda dados: dados["dep_delay"] - dados["arr_delay"],
    speed_mph=lambda dados: dados["distance"] / dados["air_time"] * 60,
)
voos_enriquecidos[["origin", "dest", "dep_delay", "arr_delay", "gain", "speed_mph"]].head()

## Selecionando colunas

Uma coluna retorna normalmente uma `Series`; uma lista de colunas retorna um `DataFrame`.

In [ ]:
df["col0"].head()

In [ ]:
df[["col0", "new_column0", "col2"]].head()

## Selecionando linhas e colunas

O primeiro elemento seleciona linhas e o segundo seleciona colunas. `.loc` usa rótulos; `.iloc` usa posições. Dois-pontos significam todas as posições daquele eixo.

In [ ]:
df.loc[["a", "b"], ["col0", "new_column0", "col2"]]

In [ ]:
df.iloc[:, [0, 1]].head()

## Selecionando por tipo

`select_dtypes()` seleciona colunas conforme o tipo. `integer` mantém inteiros; `number` inclui inteiros e números de ponto flutuante.

In [ ]:
voos.select_dtypes(include="integer").head()

## Selecionando por padrão de nome

`startswith()` produz uma máscara booleana para os nomes das colunas. Métodos semelhantes incluem `endswith()`, `contains()`, `isnumeric()` e `islower()`.

In [ ]:
df.loc[:, df.columns.str.startswith("new")].head()

## Renomeando colunas

`rename()` associa nomes antigos a novos por meio de um dicionário. Por padrão, retorna um novo DataFrame.

In [ ]:
df_renomeado = df.rename(columns={
    "col3": "letters",
    "col4": "names",
    "col6": "fruit",
})
df_renomeado.head()

## Reorganizando colunas

Uma lista explícita controla a ordem, mas omite qualquer coluna não incluída. `reindex()` é útil para ordenações sistemáticas.

In [ ]:
df_ordenado = df[["col5", "col3", "col1", "col4", "col2", "col0", "col6"]]
df_ordenado.head(3)

In [ ]:
df_ordenado.reindex(columns=sorted(df_ordenado.columns)).head(3)

# 5. Agrupamento e sumarização

`groupby()` divide os dados em grupos e `agg()` reduz várias linhas a estatísticas. A granularidade muda de voo para mês.

In [ ]:
(voos.groupby("month")[["dep_delay"]].agg("mean").head())

Agregações comuns: `count`, `size`, `first`, `last`, `mean`, `median`, `min`, `max`, `std`, `var` e `sum`.

## Múltiplas agregações

A sintaxe nomeada segue `nova_coluna=(coluna_original, função)`. `count()` conta valores não ausentes; `size()` conta linhas.

In [ ]:
atraso_mensal = (
    voos.groupby("month")
    .agg(
        atraso_medio=("dep_delay", "mean"),
        voos_com_atraso_informado=("dep_delay", "count"),
        total_voos=("flight", "size"),
    )
)
atraso_mensal.head()

## Agrupamento por múltiplas variáveis

Uma lista no `groupby()` forma grupos pela combinação das variáveis e produz um `MultiIndex`.

In [ ]:
atraso_origem_mes = (
    voos.groupby(["origin", "month"])
    .agg(atraso_medio=("dep_delay", "mean"), total_voos=("flight", "size"))
)
atraso_origem_mes.head()

## Retornando grupos para colunas

`reset_index()` transforma níveis do índice em colunas e cria um índice numérico, sem alterar as estatísticas.

In [ ]:
atraso_origem_mes.reset_index().head()

# 6. Reorganização dos dados

## Do formato largo para o longo com `melt()`

`melt()` reorganiza colunas em linhas. Colunas identificadoras são preservadas; medidas passam a formar uma coluna de identificação e outra de valores.

In [ ]:
pessoas_largo = pd.DataFrame({
    "primeiro_nome": ["John", "Mary"],
    "sobrenome": ["Doe", "Bo"],
    "profissao": ["Enfermeiro", "Economista"],
    "altura_pes": [5.5, 6.0],
    "peso_libras": [130, 150],
})
pessoas_largo

In [ ]:
pessoas_longo = pessoas_largo.melt(
    id_vars=["primeiro_nome", "sobrenome", "profissao"],
    value_vars=["altura_pes", "peso_libras"],
    var_name="medida",
    value_name="valor",
)
pessoas_longo

## Do formato longo para o largo com `pivot()`

`pivot()` usa uma coluna como índice, outra para criar novas colunas e uma terceira para preencher valores. Cada combinação deve ser única; se houver duplicidades com necessidade de agregação, use `pivot_table()`.

In [ ]:
dados_longos = pd.DataFrame({
    "data": list(pd.date_range("2000-01-01", periods=10, freq="ME")) * 2,
    "variavel": ["A"] * 10 + ["B"] * 10,
    "valor": rng.standard_normal(20),
})
dados_longos.head()

In [ ]:
dados_longos.pivot(index="data", columns="variavel", values="valor")

## `stack()` e `unstack()`

`stack()` move colunas para um nível do índice. `unstack()` move um nível do índice para colunas.

In [ ]:
tuplas = list(zip(
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
))
indice = pd.MultiIndex.from_tuples(tuplas, names=["primeiro", "segundo"])
df_multi = pd.DataFrame(rng.standard_normal((8, 2)), index=indice, columns=["A", "B"])
df_multi.head()

In [ ]:
df_empilhado = df_multi.stack().rename("valor").to_frame()
df_empilhado.head()

In [ ]:
df_empilhado["valor"].unstack(level=-1).head()

# 7. Combinação de dados

## Concatenação

`pd.concat()` combina DataFrames. `axis=0` empilha linhas; `axis=1` adiciona colunas alinhadas pelo índice. `keys` preserva a origem em um novo nível do índice.

In [ ]:
url_base = "https://github.com/aeturrell/coding-for-economists/raw/refs/heads/main/data/"
codigos_estados = ["ca", "il"]
dados_estados = [pd.read_stata(url_base + estado + "pop.dta") for estado in codigos_estados]
dados_populacao = pd.concat(dados_estados, keys=codigos_estados, axis=0)
dados_populacao

## Junções relacionais com `merge()`

`left` preserva chaves da esquerda; `right`, da direita; `inner`, apenas chaves comuns; `outer`, a união das chaves. Chaves repetidas podem multiplicar linhas.

In [ ]:
esquerda = pd.DataFrame({
    "chave1": ["K0", "K0", "K1", "K2"],
    "chave2": ["K0", "K1", "K0", "K1"],
    "A": ["A0", "A1", "A2", "A3"],
    "B": ["B0", "B1", "B2", "B3"],
})
direita = pd.DataFrame({
    "chave1": ["K0", "K1", "K1", "K2"],
    "chave2": ["K0", "K0", "K0", "K0"],
    "C": ["C0", "C1", "C2", "C3"],
    "D": ["D0", "D1", "D2", "D3"],
})
pd.merge(
    esquerda, direita,
    on=["chave1", "chave2"],
    how="right",
    validate="one_to_many",
    indicator=True,
)

A junção preserva todas as chaves da direita. Sem correspondência na esquerda, `A` e `B` recebem valores ausentes. `validate` documenta a cardinalidade esperada e `indicator=True` informa a origem de cada correspondência.

# 8. Exercícios

1. Encontre voos que:
   a) chegaram com atraso de duas horas ou mais.
   b) voaram para `IAH` ou `HOU`.
   c) foram operados por United, American ou Delta;
   d) partiram em julho, agosto ou setembro;
   e) chegaram com mais de duas horas de atraso sem partir atrasados;
   f) recuperaram mais de 30 minutos após partir com pelo menos uma hora de atraso.
2. Encontre os maiores atrasos de partida.
3. Calcule a velocidade em milhas por hora e encontre os voos mais rápidos.
4. Encontre os voos que percorreram as maiores distâncias.
5. Discuta a ordem entre `query()` e `sort_values()`.
6. Compare `air_time` com `arr_time - dep_time`. Explique por que `arr_time - dep_time` não mede diretamente a duração.
7. Compare `dep_time`, `sched_dep_time` e `dep_delay`. Como essas variaveis se relacionam?
8. Encontre e proponha diferentes formas de selecionar as colunas `dep_time`, `dep_delay`, `arr_time`, e `arr_delay`.
10. Explique `voos.columns.isin([...])`.
11. Investigue a sensibilidade a maiúsculas e minúsculas em `str.contains()`.
12. Defina duas medidas de pior atraso e compare companhias.
13. Encontre o voo mais atrasado por destino.
14. Analise os atrasos ao longo do dia.

# Referências

- [Python for Data Science: transformação de dados](https://aeturrell.github.io/python4DS/data-transform.html)
- [Python for Data Science: dados organizados](https://aeturrell.github.io/python4DS/data-tidy.html)
- [Python for Data Analysis, 3ª edição](https://wesmckinney.com/book)
- [Documentação do pandas](https://pandas.pydata.org/docs/)
- [Documentação do Matplotlib](https://matplotlib.org/stable/)
- [Documentação do seaborn](https://seaborn.pydata.org/)